# Phase 2: Forecasting Diagnostics

## The Hook
While machine learning models often beat simple benchmarks, our forecasting showdown revealed a surprising result: **The Naive Baseline outperformed LightGBM and Prophet on the 12-week holdout.**

This notebook performs a forensic diagnosis of *why* this occurred.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import sys

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

# --- HIGH PRIORITY PATH CONFIGURATION ---
cwd = os.getcwd()
project_root = os.path.abspath(os.path.join(cwd, ".." if "notebooks" in cwd else "."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.load_data import load_data
from src.data.preprocess import preprocess_data
from src.data.aggregator import aggregate_to_weekly_chain

transactions_path = os.path.join(project_root, 'data', 'raw', 'wcer.csv')
products_path = os.path.join(project_root, 'data', 'raw', 'upccer.csv')

transactions, products = load_data(transactions_path, products_path)
df = preprocess_data(transactions, products)
df_weekly = aggregate_to_weekly_chain(df)
df_weekly.tail()

## 1. The Weekly Chain-Level Trend
Let's zoom out and look at the entire 400-week history of Dominick's chain-level demand.

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(df_weekly['Week'], df_weekly['Sales'], color='#2ca02c', linewidth=1.5)
plt.title('Total Chain-Level Weekly Sales')
plt.xlabel('Week Number')
plt.ylabel('Total Units Sold')
plt.show()

## 2. The Saturation Reveal (The "Aha" Moment)
Our LightGBM model relied heavily on the `Promo` feature. But what happens if the predictive feature loses its variance?

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df_weekly['Week'], df_weekly['Promo'], color='#1f77b4', linewidth=2)

holdout_start = df_weekly['Week'].iloc[-12]
holdout_end = df_weekly['Week'].iloc[-1]
ax.axvspan(holdout_start, holdout_end, color='red', alpha=0.2, label='12-Week Holdout Period')

saturation_level = df_weekly['Promo'].iloc[-12:].mean()
ax.set_title(f'Promo Intensity - Holdout Saturation: {saturation_level:.1%}', fontweight='bold')
ax.set_xlabel('Week Number')
ax.set_ylabel('% of Stores on Promotion')
ax.legend()
plt.show()